# 실습 6: 고객용 Frontend 애플리케이션 구축

## 개요

이전 실습에서는 Memory, 공유 도구, 프로덕션 수준의 배포를 갖춘 종합적인 고객 지원 Agent를 구축했습니다. 이를 통해 Agent 사용 사례를 프로토타입에서 프로덕션으로 전환하는 AgentCore 서비스의 기능을 살펴봤습니다. 이제 모든 애플리케이션에서 Agent Runtime을 호출할 수 있습니다. 실제 애플리케이션에서 고객은 사용자 인터페이스를 기대합니다. 이제 고객이 Agent와 상호 작용할 때 실제로 사용할 수 있는 사용자 친화적인 Frontend를 만들 차례입니다.

**워크숍 진행 과정:**
- **실습 1(완료)**: Agent 프로토타입 만들기 - 작동하는 고객 지원 Agent 구축
- **실습 2(완료)**: Memory로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3(완료)**: Gateway 및 Identity로 확장 - Agent 간에 도구를 안전하게 공유
- **실습 4(완료)**: 프로덕션에 배포 - AgentCore Runtime 및 Observability 사용
- **실습 5(완료):** Agent 성능 평가 - 온라인 평가를 통해 품질 모니터링
- **실습 6(현재)**: 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기

이 실습에서는 고객이 배포된 고객 지원 Agent와 상호 작용할 수 있도록 직관적인 채팅 인터페이스를 제공하는 **Streamlit 기반 웹 애플리케이션**을 만듭니다. Frontend에는 다음 기능이 포함됩니다.

- **안전한 인증** - Amazon Cognito를 통한 사용자 로그인
- **실시간 채팅 인터페이스** - Streamlit 기반 대화형 UI
- **응답 스트리밍** - 더 나은 사용자 경험을 위한 실시간 응답 스트리밍
- **세션 관리** - Memory를 사용하는 영구 대화
- **응답 시간 측정** - 투명성을 위한 성능 지표

### 실습 6 아키텍처

Frontend 애플리케이션은 실습 4에서 배포한 AgentCore Runtime 엔드포인트에 연결되어 완전한 엔드 투 엔드 고객 지원 솔루션을 제공합니다.

<div style="text-align:left">
    <img src="images/architecture_lab6_streamlit.png" width="100%"/>
</div>


### 학습 내용

- Frontend에 안전한 인증을 통합하는 방법
- 실시간 스트리밍 응답을 구현하는 방법
- 사용자 세션 및 대화 컨텍스트를 관리하는 방법
- 고객 지원을 위한 직관적인 채팅 인터페이스를 만드는 방법

### 실습 목표

이 실습을 마치면 다음 작업을 완료하게 됩니다.

- 고객용 Streamlit 웹 애플리케이션 배포
- AgentCore Identity를 통한 안전한 사용자 인증 통합
- 실시간 스트리밍 채팅 응답 구현
- 완전한 엔드 투 엔드 고객 지원 솔루션 구축
- 로그인부터 지원 문제 해결까지 전체 고객 여정 테스트

## 사전 요구 사항

- **실습 1~4 완료**
- 로컬에 설치된 **Python 3.10 이상**
- **Streamlit** 및 필수 Frontend 종속성
- 실습 4에서 생성한 **AgentCore Runtime 엔드포인트**(배포 및 준비 완료)
- 인증용으로 구성된 **Amazon Cognito** User Pool

### 단계 1: Frontend 종속성 설치

먼저 Streamlit Frontend 애플리케이션에 필요한 패키지를 설치합니다.

In [ ]:
# Frontend 전용 종속성 설치
%pip install -r lab_helpers/lab5_frontend/requirements.txt -q
print("✅ Frontend dependencies installed successfully!")

### 단계 2: Frontend 아키텍처 이해

Streamlit 애플리케이션은 다음과 같은 주요 구성 요소로 이루어집니다.

#### 핵심 구성 요소

1. **main.py** - UI와 인증을 포함하는 기본 Streamlit 애플리케이션
2. **chat.py** - 채팅 관리 및 AgentCore Runtime 통합
3. **chat_utils.py** - 메시지 형식 지정 및 표시를 위한 유틸리티 함수
4. **sagemaker_helper.py** - 액세스 가능한 URL을 생성하는 도우미

#### 인증 흐름

1. 사용자가 Streamlit 애플리케이션에 액세스합니다.
2. Amazon Cognito가 사용자 인증을 처리합니다.
3. 유효한 JWT 토큰을 사용하여 AgentCore Runtime 요청에 권한을 부여합니다.
4. 사용자가 고객 지원 Agent와 안전하게 상호 작용할 수 있습니다.

### 단계 3: 고객 지원 Frontend 시작 🚀

이제 Streamlit 애플리케이션을 시작합니다. 애플리케이션은 다음 작업을 수행합니다.

1. 애플리케이션의 **액세스 가능한 URL 생성**
2. 포트 8501에서 **Streamlit 서버 시작**
3. 실습 4에서 **배포한 AgentCore Runtime에 연결**
4. **완전한 고객 지원 인터페이스 제공**

**중요 참고 사항:**
- 애플리케이션은 중지할 때까지 계속 실행됩니다(Ctrl+C).
- 실습 4의 AgentCore Runtime이 아직 배포되어 실행 중인지 확인하세요.
- Cognito 인증 토큰은 2시간 동안 유효합니다.

In [ ]:
# Streamlit 애플리케이션의 액세스 가능한 URL 가져오기
from lab_helpers.lab5_frontend.sagemaker_helper import get_streamlit_url

streamlit_url = get_streamlit_url()
print(f"\n🚀 Customer Support Streamlit Application URL:\n{streamlit_url}\n")

# Streamlit 애플리케이션 시작
!cd lab_helpers/lab5_frontend/ && streamlit run main.py

### 단계 4: 고객 지원 애플리케이션 테스트

Streamlit 애플리케이션이 실행되면 전체 고객 지원 경험을 테스트할 수 있습니다.

#### 인증 테스트
1. 위에 제공된 고객 지원 Streamlit 애플리케이션 URL을 사용하여 **애플리케이션에 액세스**합니다.
2. 출력에 제공된 테스트 자격 증명으로 **로그인**합니다.
3. 사용자 이름이 포함된 환영 메시지가 표시되는지 **확인**합니다.

<div style="text-align:left">
    <img src="images/lab5_streamlit_login.png"/>
</div>
<div>
    <img src="images/lab5_welcome_user.png"/>
</div>    


#### 테스트할 고객 지원 시나리오

제품 정보 질의: "What are the specifications for your laptops?"

반품 정책 질문: "What's the return policy for electronics?"

문제 해결 지원: "My iPhone is overheating, what should I do?"

<div style="text-align:left">    
    <img src="images/lab5_agent_question.png" width="75%"/>
</div>

Memory 및 개인화 테스트: 대화를 나눈 다음 페이지를 새로 고칩니다.

<div style="text-align:left">
    <img src="images/lab5_agent_chat_history.png" width="75%"/>
</div>

#### 관찰할 항목

- **실시간 스트리밍** - 응답이 생성되는 즉시 표시됩니다.
- **응답 시간** - 각 응답과 함께 성능 지표가 표시됩니다.
- **Memory 영속성** - Agent가 대화 컨텍스트를 기억합니다.
- **도구 통합** - Agent가 질의에 따라 적절한 도구를 사용합니다.
- **전문적인 UI** - 깔끔하고 직관적인 고객 지원 인터페이스를 제공합니다.
- **오류 처리** - 모든 문제를 원활하게 처리합니다.

## 🎉 실습 6 완료!

축하합니다! AI 기반 고객 지원 Agent를 위한 완전한 고객용 Frontend 애플리케이션을 성공적으로 구축하고 배포했습니다. 완료한 작업은 다음과 같습니다.

### 구축한 항목

- **웹 인터페이스** - Streamlit 기반 고객 지원 애플리케이션
- **안전한 인증** - 사용자 관리를 위한 Amazon Cognito 통합
- **실시간 스트리밍** - 더 나은 사용자 경험을 위한 실시간 응답 스트리밍
- **세션 관리** - 상호 작용 전반에서 Memory를 사용하는 영구 대화
- **완전한 통합** - AgentCore Runtime에 연결된 Frontend

### 엔드 투 엔드 고객 지원 솔루션

이제 다음 항목을 포함하는 **완전한 고객 지원 시스템**을 갖추었습니다.

1. **지능형 Agent**(실습 1) - 사용자 지정 도구를 활용하는 AI 기반 지원
2. **영구 Memory**(실습 2) - 대화 컨텍스트 및 개인화
3. **공유 도구 및 Identity**(실습 3) - 확장 가능한 도구 공유 및 액세스 제어
4. **프로덕션 Runtime**(실습 4) - Observability를 갖춘 안전하고 확장 가능한 배포
5. **고객용 Frontend**(실습 5) - 최종 사용자를 위한 웹 인터페이스

### 살펴본 주요 기능

- **멀티 턴 대화** - Agent가 상호 작용 전반에서 컨텍스트 유지
- **도구 통합** - 제품 정보, 반품 정책, 웹 검색의 원활한 사용
- **Memory 영속성** - 고객 선호도 및 이력 유지
- **실시간 성능** - 성능 지표를 포함한 응답 스트리밍
- **보안 및 Identity** - 적절한 인증 및 권한 부여
- **Observability** - Agent 동작에 대한 전체 추적 및 모니터링

### 다음 단계

고객 지원 솔루션을 더욱 강화하려면 다음 사항을 고려하세요.

- **사용자 지정 스타일** - 회사 디자인 시스템에 맞게 Frontend 브랜딩
- **추가 도구** - 기존 CRM, 티켓팅 또는 Knowledge Base 시스템과 통합
- **다국어 지원** - 전 세계 고객을 위한 국제화 추가
- **고급 분석** - 지원 팀 인사이트를 위한 사용자 지정 대시보드 구현
- **모바일 최적화** - 모바일 장치에서 인터페이스가 원활하게 작동하는지 확인

### 정리

이 워크숍에서 생성한 리소스를 정리할 준비가 되면 다음 실습으로 이동하세요.

**정리할 준비가 되었나요?** [실습 7: 정리로 이동 →](lab-07-cleanup.ipynb)

---

**🎊 Amazon Bedrock AgentCore 엔드 투 엔드 워크숍을 완료하신 것을 축하합니다!**

Amazon Bedrock AgentCore 기능을 사용하여 프로토타입부터 고객용 애플리케이션까지 완전한 프로덕션 지원 AI Agent 솔루션을 성공적으로 구축했습니다.